In [152]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
from itertools import product


#NOTAS PRELIMINARES: a base de dados do SAEB usa máscaras para os municípios e escolas.
#Assim, não é possível cruzar os dados com o Censo Escolar ou os Indicadores Educacionais,
#pois essas bases usam os identificadores originais das escolas. Para contornar isso, eu
#optei por usar as bases TS_ESCOLA e TS_PROFESSOR, disponível nos microdados do SAEB, e
#tentar aproximar as variáveis usadas pelos autores com as informações do SAEB. Por
#esse motivo, é possível que haja uma discrepância entre os resultados obtidos pelos
#autores e por essa replicação do artigo

In [46]:
#Parte 1.1: obtendo o ICSF

df_alunos_5ano = pd.read_csv('TS_ALUNO_5EF.csv', sep=',', encoding='iso8859-1')

variaveis_socioeconomicas = [
    'TX_RESP_Q005', #Televisão
    'TX_RESP_Q006', #Aparelho de rádio
    'TX_RESP_Q014', #Banheiro
    'TX_RESP_Q012', #Automóvel
    'TX_RESP_Q017', #Empregada mensalista
    'TX_RESP_Q011', #Máquina de lavar
    'TX_RESP_Q007', #Videocassete e/ou DVD
    'TX_RESP_Q008', #Geladeira
    'TX_RESP_Q009', #Freezer (parte da geladeira duplex)
    'TX_RESP_Q010'  #Freezer (separado da geladeira)
]

df_icsf = df_alunos_5ano[['ID_ALUNO'] + ['IN_PREENCHIMENTO_QUESTIONARIO'] + variaveis_socioeconomicas].copy()

correcao_valores_socioeconomicos = {
    'A':0, #Não tem
    'B':1, #Sim, um(a)
    'C':2, #Sim, dois(duas)
    'D':3, #Sim, três
    'E':4, #Sim, quatro ou mais
    '.':0, #Marcações em branco
    '*':0  #Marcações anuladas
}

for coluna in variaveis_socioeconomicas:
    df_icsf[coluna] = df_icsf[coluna].map(correcao_valores_socioeconomicos)

df_icsf['FREEZER'] = np.where((df_icsf['TX_RESP_Q009'] > 0) | (df_icsf['TX_RESP_Q010'] > 0), 1, 0)
#Os autores consideram atribuem pontuação 2 para qualquer quantidade de freezers,
#sejam eles parte ou não da geladeira. Portanto, preferi criar uma variável que
#verifica a existência de freezers ou não, que está salva na coluna 'FREEZER',
#para poder calcular o ICSF
df_icsf = df_icsf.drop(columns=['TX_RESP_Q009', 'TX_RESP_Q010'])

In [47]:
#Parte 1.2: obtendo o ICSF de cada aluno
#Realizamos a PCA para encontrar o ICSF

componentes_icsf = ['FREEZER'] + variaveis_socioeconomicas
componentes_icsf.remove('TX_RESP_Q009')
componentes_icsf.remove('TX_RESP_Q010')
#A coluna FREEZER já tem as informações dessas duas variáveis acima

df_icsf = df_icsf[df_icsf['IN_PREENCHIMENTO_QUESTIONARIO'] == 1].copy()
df_icsf = df_icsf.drop(columns=['IN_PREENCHIMENTO_QUESTIONARIO'])
#Se fizermos a PCA com os alunos que não responderam o questionário, estaremos
#subestimando o valor da média. Por isso, vamos considerar apenas os estudantes
#que responderam, pois só temos as informações deles

df_icsf[componentes_icsf] = df_icsf[componentes_icsf].fillna(0) #Não podemos ter valores NaN para fazer a PCA. Então, trocaremos eles por 0
icsf = df_icsf[componentes_icsf].values

scaler = StandardScaler()
icsf_padronizado = scaler.fit_transform(icsf)

matriz_cov_icsf = np.cov(icsf_padronizado.T)

autovalores_icsf, autovetores_icsf = np.linalg.eig(matriz_cov_icsf)

indices_icsf_organizados = np.argsort(autovalores_icsf)[::-1]
autovalores_icsf = autovalores_icsf[indices_icsf_organizados]
autovetores_icsf = autovetores_icsf[:, indices_icsf_organizados]

top_autovetor_icsf = autovetores_icsf[:, :1]

df_icsf['ICSF'] = np.dot(icsf_padronizado, top_autovetor_icsf)

if top_autovetor_icsf[0, 0] < 0:
    df_icsf['ICSF'] = -df_icsf['ICSF']
    top_autovetor_icsf = -top_autovetor_icsf

In [53]:
#Parte 2.1: gerando o dataframe para calcular o IIE
#As informações da base de dados do SAEB usam máscaras que escondem
#os códigos do INEP das escolas. Para montar o Índice de Infraestrutura
#da Escola (IIE), optei por usar variáveis parecidas com às que foram
#utilizadas na montagem do IIE original, mas utilizando a base TS_ESCOLA
#retirada do SAEB.

df_iie_bruto = pd.read_csv('TS_ESCOLA.csv', sep=',', encoding='iso-8859-1')

variaveis_infraestrutura = [
    'TX_RESP_Q018', #Abastecimento de água e existência de esgoto sanitário da rede pública: aproximado como "Instalação hidráulica"
    'TX_RESP_Q060', #Laboratório de informática
    'TX_RESP_Q061', #Laboratório de ciências
    'TX_RESP_Q059', #Quadra de esportes
    'TX_RESP_Q017', #Cozinha
    'TX_RESP_Q057', #Biblioteca
    'TX_RESP_Q049', #Televisão
    'TX_RESP_Q048', #Videocassete e DVD
    'TX_RESP_Q052', #Antena parabólica
    'TX_RESP_Q044', #Copiadora
    'TX_RESP_Q046', #Retroprojetor
    'TX_RESP_Q045', #Impressora
    'TX_RESP_Q037', #Computador
    'TX_RESP_Q053'  #Internet
]

df_iie = df_iie_bruto[['ID_ESCOLA'] + ['TX_RESP_Q066'] + variaveis_infraestrutura].copy()
#Nota: 'TX_RESP_Q066' corresponde à "Parque infantil", que foi aproximado como "Brinquedoteca"
#A Q066 contém uma formatação diferente das respostas, e precisará ser tratada individualmente

correcao_valores_iie = {
    'A':3, #Bom
    'B':2, #Regular
    'C':1, #Ruim
    'D':0, #Inexistente
    '.':0, #Marcações em branco
    '*':0  #Marcações anuladas
}

for coluna in variaveis_infraestrutura:
    df_iie[coluna] = df_iie[coluna].map(correcao_valores_iie)

correcao_valores_q066 = {
    'A':3, #Bom
    'B':2, #Regular
    'C':0, #Não há biblioteca/sala de leitura.
    '.':0, #Marcações em branco
    '*':0  #Marcações anuladas
}

df_iie['TX_RESP_Q066'] = df_iie['TX_RESP_Q066'].map(correcao_valores_q066)

In [54]:
#Parte 2.2: obtendo o IIE para cada escola
#Realizamos a PCA para encontrar o IIE

componentes_iie = ['TX_RESP_Q066'] + variaveis_infraestrutura
df_iie[componentes_iie] = df_iie[componentes_iie].fillna(0) #Não podemos ter valores NaN para fazer a PCA. Então, trocaremos eles por 0
iie = df_iie[componentes_iie].values

scaler = StandardScaler()
iie_padronizado = scaler.fit_transform(iie)

matriz_cov_iie = np.cov(iie_padronizado.T)

autovalores_iie, autovetores_iie = np.linalg.eig(matriz_cov_iie)

indices_iie_organizados = np.argsort(autovalores_iie)[::-1]
autovalores_iie = autovalores_iie[indices_iie_organizados]

autovetores_iie = autovetores_iie[:, indices_iie_organizados]

top_autovetor_iie = autovetores_iie[:, :1]

df_iie['IIE'] = np.dot(iie_padronizado, top_autovetor_iie)

if top_autovetor_iie[0, 0] < 0:
    df_iie['IIE'] = -df_iie['IIE']
    top_autovetor_iie = -top_autovetor_iie

In [118]:
#Parte 3: obtendo o IAFD para cada escola

df_iafd_bruto = pd.read_csv('TS_PROFESSOR.csv', sep=',', encoding='iso-8859-1')

variaveis_professor = [
    'ID_ESCOLA',
    'ID_SERIE',
    'TX_RESP_Q004', #Nível de escolaridade do professor
    'TX_RESP_Q009'  #Área temática do curso de pós-graduação do professor
]

df_iafd = df_iafd_bruto[variaveis_professor].copy()

df_iafd = df_iafd[df_iafd['ID_SERIE'] == 5].copy() #Estamos interessados apenas nos professores da 5ª série

correcao_valores_q004 = {
    'A':0, #Menos que o Ensino Médio (antigo 2º grau).
    'B':0, #Ensino Médio - Magistério (antigo 2º grau).
    'C':0, #Ensino  Médio - Outros (antigo 2º grau).
    'D':1, #Ensino Superior - Pedagogia.
    'E':1, #Ensino Superior - Curso Normal Superior.
    'F':1, #Ensino Superior - Licenciatura em Matemática.
    'G':1, #Ensino Superior - Licenciatura em Letras.
    'H':2, #Ensino Superior – Outras Licenciaturas.
    'I':2, #Ensino Superior - Outras áreas.
    '.':0, #Marcações em branco
    '*':0  #Marcações anuladas
}
#O IAFD usado pelos autores é composto pelo grupo 1 na categoria de adequação da formação de docentes. Esse
#é definido como "docentes com formação superior de licenciatura (ou bacharelado com complementação pedagógica)
#na mesma área da disciplina que leciona", e é representado por um valor percentual
#No dicionário acima, o valor 0 representa o professor que não se encaixa na condição do grupo 1;
#o valor 1 representa o professor que se encaixa na formação superior de licenciatura;
#e o valor 2 representa o professor que tem bacharelado com complementação pedagógica, e usaremos o dicionário
#que corrige a Q009 para verificar se ele tem a complementação correta

correcao_valores_q009 = {
    'A':0, #Não fiz ou não completei curso de pós-graduação.
    'B':1, #Educação, enfatizando alfabetização.
    'C':1, #Educação, enfatizando linguística e/ou letramento.
    'D':1, #Educação, enfatizando educação matemática.
    'E':0, #Educação - outras ênfases.
    'F':0, #Outras áreas que não a Educação.
    '.':0, #Marcações em branco
    '*':0  #Marcações anuladas
}

df_iafd['TX_RESP_Q004'] = df_iafd['TX_RESP_Q004'].map(correcao_valores_q004).fillna(0)
df_iafd['TX_RESP_Q009'] = df_iafd['TX_RESP_Q009'].map(correcao_valores_q009).fillna(0)

prof_adequado = (df_iafd['TX_RESP_Q004'] == 1) | ((df_iafd['TX_RESP_Q004'] == 2) & (df_iafd['TX_RESP_Q009'] == 1))

df_iafd['PROF_ADEQUADO'] = np.where(prof_adequado, 1, 0)

df_iafd_escolas = df_iafd.groupby('ID_ESCOLA')['PROF_ADEQUADO'].mean().reset_index()
df_iafd_escolas = df_iafd_escolas.rename(columns={'PROF_ADEQUADO': 'IAFD'})

In [174]:
#Parte 4: montando a base dos mediadores

#Definição das listas de variáveis. Elas foram separadas para poder fazer a limpeza mais facilmente depois
variaveis_frontdoor0 = [
    'ID_REGIAO',
    'ID_ESCOLA',
    'ID_LOCALIZACAO',
    'ID_ALUNO',
    'PROFICIENCIA_LP_SAEB',
    'PROFICIENCIA_MT_SAEB',
    'PESO_ALUNO_LP',
    'PESO_ALUNO_MT',
    'IN_PREENCHIMENTO_QUESTIONARIO'
]

variaveis_frontdoor1 = [
    'TX_RESP_Q001', #Sexo
    'TX_RESP_Q002', #Raça
    'TX_RESP_Q042', #Trabalha
    'TX_RESP_Q018', #Mora com a mãe
    'TX_RESP_Q022', #Mora com o pai
    'TX_RESP_Q030'  #Incentivo Escola
]

variaveis_frontdoor2 = [
    'TX_RESP_Q004' #Idade
]

variaveis_frontdoor3 = [
    'TX_RESP_Q045', #Reprovação
    'TX_RESP_Q046', #Abandono Escola
    'TX_RESP_Q013'  #Computador
]

variaveis_frontdoor4 = [
    'TX_RESP_Q047', #Dever Casa1 (LP)
    'TX_RESP_Q049', #Dever Casa2 (MT)
    'TX_RESP_Q033'  #Leitura (considera apenas livros)
]

variaveis_frontdoor5 = [
    'TX_RESP_Q023', #Educação Pai
    'TX_RESP_Q019'  #Educação Mãe
]

#Definição do dataframe que será usado para aplicar a análise de mediação. Ele é um preliminar, pois precisa passar pelo tratamento e união com o ICSF, IIE e IAFD
variaveis_frontdoor = variaveis_frontdoor0 + variaveis_frontdoor1 + variaveis_frontdoor2 + variaveis_frontdoor3 + variaveis_frontdoor4 + variaveis_frontdoor5
df_frontdoor_preliminar = df_alunos_5ano[variaveis_frontdoor].copy()

#Definição dos dicionários que tratam as variáveis
correcao_valores_frontdoor1 = {
    #Corrige as variáveis: Sexo, Raça, Trabalha, Mora com a mãe, Mora com o pai, Incentivo Escola
    'A':1,
    'B':0, 'C':0, 'D':0, 'E':0, 'F':0,
    '.':np.nan, #Marcações em branco
    '*':np.nan  #Marcações anuladas
}

correcao_valores_frontdoor2 = {
    #Corrige a variável: Idade
    'A':0,      #08 anos
    'B':1,      #09 anos
    'C':1,      #10 anos
    'D':1,      #11 anos
    'E':0,      #12 anos
    'F':0,      #13 anos
    'G':0,      #14 anos
    'H':0,      #15 anos ou mais
    '.':np.nan, #Marcações em branco
    '*':np.nan  #Marcações anuladas
}

correcao_valores_frontdoor3 = {
    #Corrige as variáveis: Reprovação, Abandono Escola, Computador
    'A':0,
    'B':1, 'C':1, 'D':1,
    '.':np.nan, #Marcações em branco
    '*':np.nan  #Marcações anuladas
}

correcao_valores_frontdoor4 = {
    #Corrige as variáveis: Dever Casa1, Dever Casa2, Leitura
    'A':1, 'B':1,
    'C':0, 'D':0,
    '.':np.nan, #Marcações em branco
    '*':np.nan  #Marcações anuladas
}

correcao_valores_frontdoor5 = {
    #Corrige as variáveis: Educação Pai, Educação Mãe
    'A':0,      #Nunca estudou
    'B':0,      #Não completou a 4a série/5o ano
    'C':0,      #Completou a 4a série/5o ano, mas não completou a 8a série/9o ano
    'D':1,      #Completou a 8a série/9o ano, mas não completou o Ensino Médio
    'E':2,      #Completou o Ensino Médio, mas não completou a Faculdade
    'F':3,      #Completou a Faculdade
    'G':np.nan, #Não sei
    '.':np.nan, #Marcações em branco
    '*':np.nan  #Marcações anuladas
}

#Aplicação do tratamento para cada uma das variáveis
for coluna in variaveis_frontdoor1:
    df_frontdoor_preliminar[coluna] = df_frontdoor_preliminar[coluna].map(correcao_valores_frontdoor1)

df_frontdoor_preliminar['TX_RESP_Q004'] = df_frontdoor_preliminar['TX_RESP_Q004'].map(correcao_valores_frontdoor2) #Correção da Idade

for coluna in variaveis_frontdoor3:
    df_frontdoor_preliminar[coluna] = df_frontdoor_preliminar[coluna].map(correcao_valores_frontdoor3)

for coluna in variaveis_frontdoor4:
    df_frontdoor_preliminar[coluna] = df_frontdoor_preliminar[coluna].map(correcao_valores_frontdoor4)

for coluna in variaveis_frontdoor5:
    df_frontdoor_preliminar[coluna] = df_frontdoor_preliminar[coluna].map(correcao_valores_frontdoor5)

#Criação das variáveis dummy de morar com ambos os pais e educação dos pais com base nas entradas
mora_com_ambos_pais = (df_frontdoor_preliminar['TX_RESP_Q018'] == 1) & (df_frontdoor_preliminar['TX_RESP_Q022'] == 1)
df_frontdoor_preliminar['MORA_PAI_MAE'] = np.where(df_frontdoor_preliminar['TX_RESP_Q018'].isna() | df_frontdoor_preliminar['TX_RESP_Q022'].isna(), np.nan, np.where(mora_com_ambos_pais, 1, 0))
#A linha acima verifica se alguma das respostas em Q018 ou Q022 é inválida, e se for, retorna a resposta inválida. Caso contrário, retorna 1 ou 0, dependendo se mora com ambos os pais ou não

df_frontdoor_preliminar['EDUCACAO_PAI1'] = np.where(df_frontdoor_preliminar['TX_RESP_Q023'].isna(), np.nan, np.where(df_frontdoor_preliminar['TX_RESP_Q023'] == 1, 1, 0))
df_frontdoor_preliminar['EDUCACAO_PAI2'] = np.where(df_frontdoor_preliminar['TX_RESP_Q023'].isna(), np.nan, np.where(df_frontdoor_preliminar['TX_RESP_Q023'] == 2, 1, 0))
df_frontdoor_preliminar['EDUCACAO_PAI3'] = np.where(df_frontdoor_preliminar['TX_RESP_Q023'].isna(), np.nan, np.where(df_frontdoor_preliminar['TX_RESP_Q023'] == 3, 1, 0))

df_frontdoor_preliminar['EDUCACAO_MAE1'] = np.where(df_frontdoor_preliminar['TX_RESP_Q019'].isna(), np.nan, np.where(df_frontdoor_preliminar['TX_RESP_Q019'] == 1, 1, 0))
df_frontdoor_preliminar['EDUCACAO_MAE2'] = np.where(df_frontdoor_preliminar['TX_RESP_Q019'].isna(), np.nan, np.where(df_frontdoor_preliminar['TX_RESP_Q019'] == 2, 1, 0))
df_frontdoor_preliminar['EDUCACAO_MAE3'] = np.where(df_frontdoor_preliminar['TX_RESP_Q019'].isna(), np.nan, np.where(df_frontdoor_preliminar['TX_RESP_Q019'] == 3, 1, 0))

#Criação do dataframe final para aplicação da análise de mediação
df_frontdoor = pd.merge( #Une df_frontdoor_preliminar e df_icsf usando a variável ID_ALUNO
    df_frontdoor_preliminar,
    df_icsf[['ID_ALUNO', 'ICSF']],
    on='ID_ALUNO',
    how='inner'
)

df_frontdoor = pd.merge(
    df_frontdoor,
    df_iie[['ID_ESCOLA', 'IIE']],
    on='ID_ESCOLA',
    how='inner'
)

df_frontdoor = pd.merge(
    df_frontdoor,
    df_iafd_escolas[['ID_ESCOLA', 'IAFD']],
    on='ID_ESCOLA',
    how='inner'
)

#Diagnóstico do Claude: o ICSF não foi padronizado no intervalo [0, 1], conforme o artigo
#As três linhas abaixo fazem essa padronização
icsf_min = df_frontdoor['ICSF'].min()
icsf_max = df_frontdoor['ICSF'].max()
df_frontdoor['ICSF'] = (df_frontdoor['ICSF'] - icsf_min) / (icsf_max - icsf_min)

#Vamos mudar df_frontdoor para considerar apenas os estudantes de áreas urbanas
df_frontdoor = df_frontdoor[df_frontdoor['ID_LOCALIZACAO'] == 1].copy()

In [167]:
# -----------------------------------------------------------------------------
# BLOCO 1: FUNÇÕES DO ALGORITMO DE MEDIAÇÃO
# -----------------------------------------------------------------------------
 
def causal_mediation(df, treatment, mediator, outcome, controls,
                     weights_col=None, n_boot=1000, conf_level=0.95,
                     random_state=42):
    """
    Estimação de efeitos de mediação causal seguindo Imai et al. (2010).
 
    Parâmetros
    ----------
    df          : DataFrame já filtrado e sem NaN nas colunas usadas
    treatment   : str — variável de tratamento binária (0/1)
    mediator    : str — variável mediadora (ICSF)
    outcome     : str — variável de resultado (PROFICIENCIA_LP_SAEB ou MT)
    controls    : list[str] — covariadas de pré-tratamento
    weights_col : str ou None — coluna de pesos amostrais
    n_boot      : int — replicações bootstrap (use 20 para testes, 1000 para final)
    conf_level  : float — nível de confiança
    random_state: int — semente
 
    Retorna
    -------
    dict com estimativas pontuais e intervalos de confiança bootstrap
    """
    rng = np.random.default_rng(random_state)
    str_controls = " + ".join(controls)
 
    formula_m = f"{mediator} ~ {treatment} + {str_controls}"
    formula_y = f"{outcome} ~ {treatment} + {mediator} + {str_controls}"
 
    def _fit_models(data):
        w = data[weights_col].values if weights_col else None
        mod_m = smf.wls(formula_m, data=data, weights=w).fit(disp=False)
        mod_y = smf.wls(formula_y, data=data, weights=w).fit(disp=False)
        return mod_m, mod_y
 
    def _compute_effects(data, mod_m, mod_y):
        # Cópias contrafactuais com T fixado
        d0 = data.copy(); d0[treatment] = 0
        d1 = data.copy(); d1[treatment] = 1
 
        # Mediador predito sob T=0 e T=1
        m0 = mod_m.predict(d0)
        m1 = mod_m.predict(d1)
 
        # ACME: δ(t) = E[Y(t, M(1))] - E[Y(t, M(0))]
        d1_m1 = d1.copy(); d1_m1[mediator] = m1
        d1_m0 = d1.copy(); d1_m0[mediator] = m0
        d0_m1 = d0.copy(); d0_m1[mediator] = m1
        d0_m0 = d0.copy(); d0_m0[mediator] = m0
 
        acme_t1  = (mod_y.predict(d1_m1) - mod_y.predict(d1_m0)).mean()
        acme_t0  = (mod_y.predict(d0_m1) - mod_y.predict(d0_m0)).mean()
        acme_avg = 0.5 * (acme_t1 + acme_t0)
 
        # ADE: ζ(t) = E[Y(1, M(t))] - E[Y(0, M(t))]
        ade_t1  = (mod_y.predict(d1_m1) - mod_y.predict(d0_m1)).mean()
        ade_t0  = (mod_y.predict(d1_m0) - mod_y.predict(d0_m0)).mean()
        ade_avg = 0.5 * (ade_t1 + ade_t0)
 
        total = acme_avg + ade_avg
        prop  = acme_avg / total if total != 0 else np.nan
 
        return {
            'ACME_treated': acme_t1, 'ACME_control': acme_t0, 'ACME_avg': acme_avg,
            'ADE_treated':  ade_t1,  'ADE_control':  ade_t0,  'ADE_avg':  ade_avg,
            'Total_effect': total,   'Prop_mediated': prop,
        }
 
    # Estimativa pontual
    mod_m, mod_y = _fit_models(df)
    point = _compute_effects(df, mod_m, mod_y)
 
    # Bootstrap
    keys = list(point.keys())
    boot = {k: [] for k in keys}
    n = len(df)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        df_b = df.iloc[idx].reset_index(drop=True)
        try:
            res_b = _compute_effects(df_b, *_fit_models(df_b))
            for k in keys:
                boot[k].append(res_b[k])
        except Exception:
            pass
 
    alpha = 1 - conf_level
    ci = {k: (np.nanpercentile(boot[k], 100 * alpha / 2),
               np.nanpercentile(boot[k], 100 * (1 - alpha / 2)))
          for k in keys}
 
    return {**point, 'ci': ci}
 
 
def print_mediation_summary(result, label=""):
    ci = result['ci']
    print(f"\n{'='*65}")
    if label:
        print(f"  {label}")
    print(f"{'='*65}")
    print(f"{'Efeito':<30} {'Estimativa':>12} {'IC inf':>10} {'IC sup':>10}")
    print(f"{'-'*65}")
    for nome, key in [
        ("ACME - tratados [δ(1)]",  'ACME_treated'),
        ("ACME - controles [δ(0)]", 'ACME_control'),
        ("ACME - média",            'ACME_avg'),
        ("ADE - tratados [ζ(1)]",   'ADE_treated'),
        ("ADE - controles [ζ(0)]",  'ADE_control'),
        ("ADE - média",             'ADE_avg'),
        ("Efeito Total [τ]",        'Total_effect'),
        ("Proporção Mediada",       'Prop_mediated'),
    ]:
        v = result[key]
        lo, hi = ci[key]
        print(f"{nome:<30} {v:>12.4f} {lo:>10.4f} {hi:>10.4f}")
    print(f"{'='*65}\n")


In [168]:
# -----------------------------------------------------------------------------
# BLOCO 2: FUNÇÃO AUXILIAR — cria subamostra correta por nível de escolaridade
# -----------------------------------------------------------------------------
 
def criar_subamostra_tratamento(df, col_escolaridade, nivel_tratado):
    """
    Replica a lógica da seção 3.3 do artigo:
    - Tratados : pais com exatamente `nivel_tratado`
    - Controle : pais com escolaridade ABAIXO de `nivel_tratado`
    - Excluídos: pais com escolaridade ACIMA de `nivel_tratado`
 
    Níveis definidos no correcao_valores_frontdoor5:
        0 = abaixo do fundamental completo
        1 = fundamental completo
        2 = médio completo
        3 = superior completo
    """
    df_sub = df[df[col_escolaridade].isin(range(0, nivel_tratado + 1))].copy()
    df_sub['TRATAMENTO'] = (df_sub[col_escolaridade] == nivel_tratado).astype(int)
    return df_sub

In [169]:
# -----------------------------------------------------------------------------
# BLOCO 3: CONFIGURAÇÕES DA ANÁLISE
# -----------------------------------------------------------------------------
 
CONTROLES = [
    'TX_RESP_Q001',  # Sexo
    'TX_RESP_Q002',  # Raça
    'TX_RESP_Q004',  # Idade
    'TX_RESP_Q045',  # Reprovação
    'TX_RESP_Q046',  # Abandono Escola
    'TX_RESP_Q047',  # Dever Casa LP
    'TX_RESP_Q049',  # Dever Casa MT
    'TX_RESP_Q042',  # Trabalha
    'TX_RESP_Q033',  # Leitura
    'TX_RESP_Q013',  # Computador
    'MORA_PAI_MAE',  # Mora com pai e mãe
    'TX_RESP_Q030',  # Incentivo escolar
    'IIE',           # Índice Infraestrutura da Escola
    'IAFD',          # Índice Adequação Formação Docente
]
 
# Cada entrada: (nome_legível, col_escolaridade, nivel_tratado)
TRATAMENTOS = [
    ('PAI_FUNDAMENTAL', 'TX_RESP_Q023', 1),
    ('PAI_MEDIO',       'TX_RESP_Q023', 2),
    ('PAI_SUPERIOR',    'TX_RESP_Q023', 3),
    ('MAE_FUNDAMENTAL', 'TX_RESP_Q019', 1),
    ('MAE_MEDIO',       'TX_RESP_Q019', 2),
    ('MAE_SUPERIOR',    'TX_RESP_Q019', 3),
]
 
# Cada entrada: (nome_legível, col_outcome, col_peso)
OUTCOMES = [
    ('LP', 'PROFICIENCIA_LP_SAEB', 'PESO_ALUNO_LP'),
    ('MT', 'PROFICIENCIA_MT_SAEB', 'PESO_ALUNO_MT'),
]
 
# Regiões conforme ID_REGIAO do SAEB
REGIOES = {
    'Norte':        1,
    'Nordeste':     2,
    'Sudeste':      3,
    'Sul':          4,
    'Centro-Oeste': 5,
}
 
# Gênero: TX_RESP_Q001 já foi binarizado na Parte 4 (1=masculino, 0=feminino)
GENEROS = {
    'Filhos': 1,
    'Filhas': 0,
}
 
# Número de replicações bootstrap
# Use 20 para testes rápidos; 1000 para a análise final
N_BOOT = 1000

In [170]:
# -----------------------------------------------------------------------------
# BLOCO 4: FUNÇÃO PRINCIPAL — roda todas as combinações
# -----------------------------------------------------------------------------
 
def run_all_mediation_analyses(df_frontdoor, n_boot=N_BOOT):
    """
    Roda a mediação causal para todas as combinações de:
        6 tratamentos × 2 outcomes × 5 regiões × 2 gêneros = 120 análises
 
    Retorna um DataFrame com todos os resultados.
    """
    all_results = []
    total = len(TRATAMENTOS) * len(OUTCOMES) * len(REGIOES) * len(GENEROS)
    n_atual = 0
 
    for nome_trat, col_esc, nivel in TRATAMENTOS:
 
        # Criar subamostra para este tratamento (exclui níveis acima)
        df_trat = criar_subamostra_tratamento(df_frontdoor, col_esc, nivel)
 
        for nome_out, col_out, col_peso in OUTCOMES:
            for nome_regiao, cod_regiao in REGIOES.items():
                for nome_genero, cod_genero in GENEROS.items():
 
                    n_atual += 1
                    label = f"{nome_regiao} | {nome_genero} | {nome_trat} | {nome_out}"
                    print(f"[{n_atual}/{total}] {label}")
 
                    # Filtro de região e gênero
                    mask = (
                        (df_trat['ID_REGIAO'] == cod_regiao) &
                        (df_trat['TX_RESP_Q001'] == cod_genero)
                    )
                    df_sub = df_trat[mask].copy()
 
                    # Selecionar apenas colunas necessárias e remover NaN
                    cols_needed = (
                        ['TRATAMENTO', 'ICSF', col_out, col_peso]
                        + CONTROLES
                    )
                    cols_ok = [c for c in cols_needed if c in df_sub.columns]
                    df_modelo = df_sub[cols_ok].dropna().copy()
 
                    if len(df_modelo) < 100:
                        print(f"  ⚠ Amostra insuficiente ({len(df_modelo)} obs). Pulando.")
                        continue
 
                    controles_ok = [c for c in CONTROLES if c in df_modelo.columns]
 
                    try:
                        res = causal_mediation(
                            df=df_modelo,
                            treatment='TRATAMENTO',
                            mediator='ICSF',
                            outcome=col_out,
                            controls=controles_ok,
                            weights_col=col_peso,
                            n_boot=n_boot,
                            conf_level=0.95,
                            random_state=42,
                        )
 
                        # Montar linha do resultado
                        row = {
                            'Regiao':     nome_regiao,
                            'Genero':     nome_genero,
                            'Tratamento': nome_trat,
                            'Outcome':    nome_out,
                            'N':          len(df_modelo),
                        }
                        for k in ['ACME_treated', 'ACME_control', 'ACME_avg',
                                  'ADE_treated',  'ADE_control',  'ADE_avg',
                                  'Total_effect', 'Prop_mediated']:
                            row[k]           = res[k]
                            row[f'{k}_ci_lo'] = res['ci'][k][0]
                            row[f'{k}_ci_hi'] = res['ci'][k][1]
 
                        all_results.append(row)
                        print_mediation_summary(res, label=label)
 
                    except Exception as e:
                        print(f"  ✗ Erro: {e}")
 
    df_results = pd.DataFrame(all_results)
    return df_results

In [175]:
# -----------------------------------------------------------------------------
# BLOCO 5: EXECUÇÃO
# -----------------------------------------------------------------------------
# Para um teste rápido de uma combinação antes de rodar tudo:
#
#   df_pai1 = criar_subamostra_tratamento(df_frontdoor, 'TX_RESP_Q023', 1)
#   cols = ['TRATAMENTO', 'ICSF', 'PROFICIENCIA_MT_SAEB', 'PESO_ALUNO_MT'] + CONTROLES
#   cols = [c for c in cols if c in df_pai1.columns]
#   df_modelo = df_pai1[cols].dropna().copy()
#   controles_ok = [c for c in CONTROLES if c in df_modelo.columns]
#   res = causal_mediation(df_modelo, 'TRATAMENTO', 'ICSF', 'PROFICIENCIA_MT_SAEB',
#                          controles_ok, 'PESO_ALUNO_MT', n_boot=20)
#   print_mediation_summary(res, 'TESTE RAPIDO: PAI_FUNDAMENTAL | MT | Geral')
#
# Para rodar a análise completa (recomenda-se n_boot=1000 para o resultado final):
 
df_resultados = run_all_mediation_analyses(df_frontdoor, n_boot=20)

df_resultados.to_csv('resultados_mediacao_causal.csv', index=False, encoding='utf-8-sig')
print(f"\n✓ Concluído! {len(df_resultados)} análises salvas em 'resultados_mediacao_causal.csv'")
print(df_resultados)

[1/120] Norte | Filhos | PAI_FUNDAMENTAL | LP

  Norte | Filhos | PAI_FUNDAMENTAL | LP
Efeito                           Estimativa     IC inf     IC sup
-----------------------------------------------------------------
ACME - tratados [δ(1)]               0.3641     0.2530     0.4655
ACME - controles [δ(0)]              0.3641     0.2530     0.4655
ACME - média                         0.3641     0.2530     0.4655
ADE - tratados [ζ(1)]                3.1659     2.3154     4.3964
ADE - controles [ζ(0)]               3.1659     2.3154     4.3964
ADE - média                          3.1659     2.3154     4.3964
Efeito Total [τ]                     3.5300     2.6623     4.8026
Proporção Mediada                    0.1031     0.0712     0.1412

[2/120] Norte | Filhas | PAI_FUNDAMENTAL | LP

  Norte | Filhas | PAI_FUNDAMENTAL | LP
Efeito                           Estimativa     IC inf     IC sup
-----------------------------------------------------------------
ACME - tratados [δ(1)]           